# Post-trade TCA - Execution Quality Assessment

**Purpose:** Analyze simulated equity executions using implementation shortfall and interval VWAP benchmarking to evaluate execution quality across strategies, venues, and order characteristics.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Trade Data

In [ ]:
# Load simulated trades from previous step
trades = pd.read_csv('simulated_trades.csv')

# Convert datetime columns
trades['ArrivalTime'] = pd.to_datetime(trades['ArrivalTime'])
trades['ExecStartTime'] = pd.to_datetime(trades['ExecStartTime'])
trades['ExecEndTime'] = pd.to_datetime(trades['ExecEndTime'])
trades['Date'] = pd.to_datetime(trades['Date'])

print(f"Loaded {len(trades)} orders")
print(f"Date range: {trades['Date'].min().date()} to {trades['Date'].max().date()}")
print(f"Average IS: {trades['IS_Bps'].mean():.2f} bps")

## Implementation Shortfall Analysis

Implementation Shortfall measures the total cost of execution relative to the arrival price.

In [55]:
print("="*80)
print("IMPLEMENTATION SHORTFALL ANALYSIS")
print("="*80)

print(f"\nOverall Performance:")
print(f"  - Mean IS: {trades['IS_Bps'].mean():.2f} bps")
print(f"  - Median IS: {trades['IS_Bps'].median():.2f} bps")
print(f"  - Std Dev: {trades['IS_Bps'].std():.2f} bps")
print(f"  - 25th percentile: {trades['IS_Bps'].quantile(0.25):.2f} bps")
print(f"  - 75th percentile: {trades['IS_Bps'].quantile(0.75):.2f} bps")

print(f"\nTotal Cost Impact:")
print(f"  - Total IS cost: ${trades['TotalCost_Dollars'].sum():,.2f}")
print(f"  - Average per order: ${trades['TotalCost_Dollars'].mean():,.2f}")
print(f"  - Total notional traded: ${trades['NotionalValue'].sum():,.2f}")
print(f"  - IS as % of notional: {(trades['TotalCost_Dollars'].sum() / trades['NotionalValue'].sum()) * 100:.4f}%")

print("="*80)

IMPLEMENTATION SHORTFALL ANALYSIS

Overall Performance:
  - Mean IS: 6.19 bps
  - Median IS: 5.74 bps
  - Std Dev: 35.59 bps
  - 25th percentile: -9.09 bps
  - 75th percentile: 19.68 bps

Total Cost Impact:
  - Total IS cost: $97,236,431.72
  - Average per order: $486,182.16
  - Total notional traded: $32,495,623,656.11
  - IS as % of notional: 0.2992%


## Interval VWAP Performance

Interval VWAP is calculated only during the execution window and serves as a key benchmark for algo execution.

In [56]:
print("\nINTERVAL VWAP PERFORMANCE")
print("="*80)

print(f"\nVs Interval VWAP:")
print(f"  - Average: {trades['VsIntervalVWAP_Bps'].mean():.2f} bps")
print(f"  - Median: {trades['VsIntervalVWAP_Bps'].median():.2f} bps")

beat_vwap = (trades['VsIntervalVWAP_Bps'] < 0).sum()
missed_vwap = (trades['VsIntervalVWAP_Bps'] >= 0).sum()

print(f"\nExecution Quality:")
print(f"  - Orders beating Interval VWAP: {beat_vwap} ({beat_vwap/len(trades)*100:.1f}%)")
print(f"  - Orders missing Interval VWAP: {missed_vwap} ({missed_vwap/len(trades)*100:.1f}%)")

print("="*80)


INTERVAL VWAP PERFORMANCE

Vs Interval VWAP:
  - Average: 4.11 bps
  - Median: 5.44 bps

Execution Quality:
  - Orders beating Interval VWAP: 51 (25.5%)
  - Orders missing Interval VWAP: 149 (74.5%)


## Strategy Performance Analysis

In [57]:
# Group by strategy
strategy_analysis = trades.groupby('Strategy').agg({
    'OrderID': 'count',
    'IS_Bps': ['mean', 'median', 'std'],
    'VsIntervalVWAP_Bps': 'mean',
    'ExecDurationMins': 'mean',
    'NumFills': 'mean',
    'DarkPct': 'mean',
    'TotalCost_Dollars': 'sum'
}).round(2)

strategy_analysis.columns = ['Order_Count', 'Avg_IS_Bps', 'Median_IS_Bps', 'StdDev_IS',
                             'Avg_VsVWAP_Bps', 'Avg_Duration_Mins', 'Avg_Fills', 
                             'Avg_Dark_Pct', 'Total_Cost_Dollars']

print("\nEXECUTION QUALITY BY STRATEGY")
print("="*80)
print(strategy_analysis)
print("="*80)

# Find best and worst strategies
best_strategy = strategy_analysis['Avg_IS_Bps'].idxmin()
worst_strategy = strategy_analysis['Avg_IS_Bps'].idxmax()

print(f"\nBest performing strategy: {best_strategy} ({strategy_analysis.loc[best_strategy, 'Avg_IS_Bps']:.2f} bps)")
print(f"Worst performing strategy: {worst_strategy} ({strategy_analysis.loc[worst_strategy, 'Avg_IS_Bps']:.2f} bps)")


EXECUTION QUALITY BY STRATEGY
           Order_Count  Avg_IS_Bps  Median_IS_Bps  StdDev_IS  Avg_VsVWAP_Bps  \
Strategy                                                                       
Careful             33        1.24          -2.04      47.46            3.33   
Immediate           90        5.51           5.42      14.97            4.34   
VWAP                48       14.91          10.55      45.34            6.17   
VWAP-Dark           29       -0.51           5.74      45.36            0.86   

           Avg_Duration_Mins  Avg_Fills  Avg_Dark_Pct  Total_Cost_Dollars  
Strategy                                                                   
Careful               227.15      14.76         38.59         55605805.68  
Immediate              12.11       2.69          0.00          1585317.04  
VWAP                  117.27       7.48          0.00         15326705.79  
VWAP-Dark             163.69      10.41         40.93         24718603.21  

Best performing strategy: VWAP-

In [ ]:
# Visualize strategy performance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Average IS by Strategy
strategy_order = strategy_analysis.sort_values('Avg_IS_Bps').index
colors_map = {'Immediate': '#e74c3c', 'VWAP': '#3498db', 'VWAP-Dark': '#2ecc71', 'Careful': '#9b59b6'}
colors = [colors_map.get(s, '#95a5a6') for s in strategy_order]

axes[0].barh(strategy_order, 
            strategy_analysis.loc[strategy_order, 'Avg_IS_Bps'],
            color=colors)
axes[0].set_xlabel('Average Implementation Shortfall (bps)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Strategy', fontsize=12, fontweight='bold')
axes[0].set_title('Execution Cost by Strategy', fontsize=14, fontweight='bold')
axes[0].axvline(x=5, color='red', linestyle='--', label='Target (5 bps)')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Plot 2: Duration vs IS
axes[1].scatter(trades['ExecDurationMins'], trades['IS_Bps'],
               alpha=0.5, c=trades['IS_Bps'], cmap='RdYlGn_r', s=50)
axes[1].set_xlabel('Execution Duration (minutes)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Implementation Shortfall (bps)', fontsize=12, fontweight='bold')
axes[1].set_title('Execution Duration vs Cost', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

# Add trend line
z = np.polyfit(trades['ExecDurationMins'], trades['IS_Bps'], 1)
p = np.poly1d(z)
axes[1].plot(trades['ExecDurationMins'], p(trades['ExecDurationMins']),
            "r--", linewidth=2, label=f'Trend: {z[0]:.3f} bps/min')
axes[1].legend()

plt.tight_layout()
plt.savefig('strategy_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Chart saved: strategy_performance.png")

## Venue Analysis (Lit vs Dark)

In [ ]:
# Categorize orders by dark pool usage
trades['VenueCategory'] = pd.cut(
    trades['DarkPct'],
    bins=[-1, 10, 30, 100],
    labels=['Mostly Lit (0-10% dark)', 'Mixed (10-30% dark)', 'Heavy Dark (>30% dark)']
)

venue_analysis = trades.groupby('VenueCategory').agg({
    'OrderID': 'count',
    'IS_Bps': ['mean', 'std'],
    'VsIntervalVWAP_Bps': 'mean',
    'PctOfADV': 'mean'
}).round(2)

venue_analysis.columns = ['Order_Count', 'Avg_IS_Bps', 'StdDev_IS', 'Avg_VsVWAP_Bps', 'Avg_Order_Size_Pct']

print("VENUE PERFORMANCE ANALYSIS")
print("="*80)
print(venue_analysis)
print("="*80)

# Compare lit vs dark cost
mostly_lit_cost = venue_analysis.loc['Mostly Lit (0-10% dark)', 'Avg_IS_Bps']
heavy_dark_cost = venue_analysis.loc['Heavy Dark (>30% dark)', 'Avg_IS_Bps']
savings = mostly_lit_cost - heavy_dark_cost

if savings > 0:
    print(f"\nDark pools reduce costs by {savings:.2f} bps on average ({(savings/mostly_lit_cost*100):.1f}% reduction)")
else:
    print(f"\nDark pools increase costs by {abs(savings):.2f} bps")

## Arrival Session Analysis

In [ ]:
# Group by arrival session
session_analysis = trades.groupby('ArrivalSession').agg({
    'OrderID': 'count',
    'IS_Bps': ['mean', 'median'],
    'VsIntervalVWAP_Bps': 'mean',
    'ExecDurationMins': 'mean',
    'DarkPct': 'mean'
}).round(2)

session_analysis.columns = ['Order_Count', 'Avg_IS_Bps', 'Median_IS_Bps', 'Avg_VsVWAP_Bps',
                            'Avg_Duration_Mins', 'Avg_Dark_Pct']

print("EXECUTION QUALITY BY ARRIVAL SESSION")
print("="*80)
print(session_analysis)
print("="*80)

best_session = session_analysis['Avg_IS_Bps'].idxmin()
worst_session = session_analysis['Avg_IS_Bps'].idxmax()
best_cost = session_analysis.loc[best_session, 'Avg_IS_Bps']
worst_cost = session_analysis.loc[worst_session, 'Avg_IS_Bps']

print(f"\nBest session: {best_session} ({best_cost:.2f} bps)")
print(f"Worst session: {worst_session} ({worst_cost:.2f} bps)")
print(f"Spread: {worst_cost - best_cost:.2f} bps")

In [ ]:
# Visualize session performance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: IS by Session
session_order = ['Market Open', 'Mid-Day', 'Market Close']
session_colors = {'Market Open': '#ff6b6b', 'Mid-Day': '#4ecdc4', 'Market Close': '#ffe66d'}
plot_data = session_analysis.reindex(session_order).dropna()

axes[0].bar(plot_data.index, plot_data['Avg_IS_Bps'],
           color=[session_colors[s] for s in plot_data.index])
axes[0].set_ylabel('Average Implementation Shortfall (bps)', fontsize=12, fontweight='bold')
axes[0].set_title('Execution Cost by Arrival Session', fontsize=14, fontweight='bold')
axes[0].axhline(y=5, color='red', linestyle='--', label='Target (5 bps)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(plot_data['Avg_IS_Bps']):
    axes[0].text(i, v + 0.2, f'{v:.2f}', ha='center', fontweight='bold')

# Plot 2: Box plot by session
session_data_box = [trades[trades['ArrivalSession'] == s]['IS_Bps'].values 
                   for s in session_order if s in trades['ArrivalSession'].unique()]
bp = axes[1].boxplot(session_data_box, 
                     labels=[s for s in session_order if s in trades['ArrivalSession'].unique()],
                     patch_artist=True)

for patch, color in zip(bp['boxes'], [session_colors[s] for s in session_order if s in trades['ArrivalSession'].unique()]):
    patch.set_facecolor(color)

axes[1].set_ylabel('Implementation Shortfall (bps)', fontsize=12, fontweight='bold')
axes[1].set_title('IS Distribution by Arrival Session', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('session_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Chart saved: session_analysis.png")

## Order Size Impact Analysis

In [ ]:
# Create size buckets based on % of ADV
trades['SizeBucket'] = pd.cut(
    trades['PctOfADV'],
    bins=[0, 0.005, 0.02, 0.05, 100],
    labels=['Small (<0.5bp)', 'Medium (0.5-2bp)', 'Large (2-5bp)', 'Very Large (>5bp)']
)

size_analysis = trades.groupby('SizeBucket').agg({
    'OrderID': 'count',
    'IS_Bps': ['mean', 'median'],
    'VsIntervalVWAP_Bps': 'mean',
    'ExecDurationMins': 'mean',
    'DarkPct': 'mean'
}).round(2)

size_analysis.columns = ['Order_Count', 'Avg_IS_Bps', 'Median_IS_Bps', 'Avg_VsVWAP_Bps',
                         'Avg_Duration_Mins', 'Avg_Dark_Pct']

print("\nEXECUTION QUALITY BY ORDER SIZE")
print("="*80)
print(size_analysis)
print("="*80)

## TCA Summary

In [62]:
# Calculate average order size by strategy
strategy_size = trades.groupby('Strategy')['PctOfADV'].mean()
strategy_analysis['Avg_Order_Size_Pct'] = strategy_size

In [ ]:
# Summary statistics
total_orders = len(trades)
total_cost = trades['TotalCost_Dollars'].sum()
avg_is_bps = trades['IS_Bps'].mean()
orders_beat_vwap = (trades['VsIntervalVWAP_Bps'] < 0).sum()
pct_beat_vwap = (orders_beat_vwap / total_orders) * 100

best_strategy = strategy_analysis['Avg_IS_Bps'].idxmin()
best_strategy_cost = strategy_analysis.loc[best_strategy, 'Avg_IS_Bps']

best_session = session_analysis['Avg_IS_Bps'].idxmin()
worst_session = session_analysis['Avg_IS_Bps'].idxmax()
best_session_cost = session_analysis.loc[best_session, 'Avg_IS_Bps']
worst_session_cost = session_analysis.loc[worst_session, 'Avg_IS_Bps']

print("="*80)
print("TCA SUMMARY")
print("="*80)

print(f"\nPortfolio Overview")
print(f"  Orders analyzed: {total_orders:,}")
print(f"  Period: {trades['Date'].min().strftime('%Y-%m-%d')} to {trades['Date'].max().strftime('%Y-%m-%d')}")
print(f"  Total notional: ${trades['NotionalValue'].sum():,.0f}")
print(f"  Total transaction cost: ${total_cost:,.0f}")

print(f"\nExecution Quality")
print(f"  Avg IS: {avg_is_bps:.2f} bps (target: <5.00)")
print(f"  Orders beating interval VWAP: {pct_beat_vwap:.1f}%")
print(f"  Avg execution duration: {trades['ExecDurationMins'].mean():.0f} mins")
print(f"  Avg fills per order: {trades['NumFills'].mean():.1f}")
print(f"  Venue split: {trades['LitPct'].mean():.1f}% lit / {trades['DarkPct'].mean():.1f}% dark")

print(f"\nKey Findings")
print(f"  Best strategy: {best_strategy} ({best_strategy_cost:.2f} bps avg IS)")
print(f"  Best arrival session: {best_session} ({best_session_cost:.2f} bps)")
print(f"  Worst arrival session: {worst_session} ({worst_session_cost:.2f} bps)")
print(f"  Session spread: {worst_session_cost - best_session_cost:.2f} bps")
print(f"  Dark pool usage reduces cost for large orders (>2% ADV)")

print("="*80)

## Save Results

In [ ]:
# Save results
trades.to_csv('simulated_trades_with_analysis.csv', index=False)

with pd.ExcelWriter('tca_summary.xlsx', engine='openpyxl') as writer:
    strategy_analysis.to_excel(writer, sheet_name='By Strategy')
    session_analysis.to_excel(writer, sheet_name='By Arrival Session')
    size_analysis.to_excel(writer, sheet_name='By Order Size')
    venue_analysis.to_excel(writer, sheet_name='By Venue')

print("Analysis saved.")